# Feature construction and feature splitting
Rebuilt for pandas 3 and scikit-learn 1.9. Data: `data/titanic.csv`, the 891-passenger Titanic training file (full data: the Kaggle competition *Titanic: Machine Learning from Disaster*).

In [ ]:
# libraries for tables, models and charts
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## 1. Feature construction: family type

In [ ]:
# keep four inputs and the target
df = pd.read_csv('data/titanic.csv')[['Age', 'Pclass', 'SibSp', 'Parch', 'Survived']]
# drop the 177 passengers with no age
df = df.dropna()
df.shape

In [ ]:
# inputs X and target y
X = df.drop(columns='Survived')
y = df['Survived']
X.head()

In [ ]:
# baseline: mean accuracy over 20 folds (the same setup as the original code)
lr = LogisticRegression(max_iter=1000)
cross_val_score(lr, X, y, cv=20).mean()

In [ ]:
# family size: siblings/spouses + parents/children + the passenger
X_new = X.copy()
X_new['Family_size'] = X_new['SibSp'] + X_new['Parch'] + 1

# group the size: 0 = alone, 1 = small (2 to 4), 2 = large (5 or more)
size = X_new['Family_size']
X_new['Family_type'] = np.select([size == 1, size <= 4], [0, 1], 2)
X_new.head()

In [ ]:
# survival rate for each family type: not a straight line
df.groupby(X_new['Family_type'])['Survived'].agg(['mean', 'size'])

In [ ]:
# keep Family_type, drop the columns it was built from
X_new = X_new.drop(columns=['SibSp', 'Parch', 'Family_size'])
cross_val_score(lr, X_new, y, cv=20).mean()

20 folds give a noisy number. Below we repeat 10-fold cross-validation 10 times (100 scores) and build every new column **inside a pipeline**, so each fold builds it from its own training rows.

In [ ]:
# one function builds every new column from the raw columns
def add_columns(X):
    X = X.copy()
    X['Family_size'] = X['SibSp'] + X['Parch'] + 1
    size = X['Family_size']
    X['Family_type'] = np.select([size == 1, size <= 4], [0, 1], 2)
    # title: the text between ', ' and the first '.'
    X['Title'] = X['Name'].str.split(', ').str[1].str.split('.').str[0]
    X['Is_Married'] = (X['Title'] == 'Mrs').astype(int)
    return X

In [ ]:
# a model = build columns -> keep (and one-hot encode) some -> logistic regression
def model(cols, onehot=()):
    pick = make_column_transformer(
        # titles seen fewer than 10 times are grouped into one 'infrequent' column
        (OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10), list(onehot)),
        ('passthrough', [c for c in cols if c not in onehot]))
    return make_pipeline(FunctionTransformer(add_columns), pick, LogisticRegression(max_iter=1000))

In [ ]:
# the same 714 passengers for every comparison
raw = pd.read_csv('data/titanic.csv').dropna(subset=['Age'])
Xr = raw[['Age', 'Pclass', 'SibSp', 'Parch', 'Name']]
yr = raw['Survived']
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=0)

BASE = ['Age', 'Pclass', 'SibSp', 'Parch']
sets = {
    'baseline': model(BASE),
    '+ Family_size': model(BASE + ['Family_size']),
    'Family_type 0/1/2': model(['Age', 'Pclass', 'Family_type']),
    'Family_type one-hot': model(['Age', 'Pclass', 'Family_type'], ['Family_type']),
    '+ Is_Married': model(BASE + ['Is_Married']),
    '+ Title one-hot': model(BASE + ['Title'], ['Title']),
}
scores = {name: cross_val_score(m, Xr, yr, cv=cv) for name, m in sets.items()}
pd.DataFrame({'mean': [s.mean() for s in scores.values()],
              'std': [s.std() for s in scores.values()]}, index=scores.keys()).round(4)

In [ ]:
# paired check: in how many of the 100 folds does a column beat the baseline?
for name in ['Family_type 0/1/2', '+ Title one-hot']:
    d = scores[name] - scores['baseline']
    print(name, 'better:', (d > 0).sum(), 'worse:', (d < 0).sum(), 'tied:', (d == 0).sum())

## 2. Feature splitting: the title inside the name

In [ ]:
df = pd.read_csv('data/titanic.csv')
df['Name'].head()

In [ ]:
# step 1: split at ', ' and keep piece 1; step 2: split that at '.' and keep piece 0
df['Title'] = df['Name'].str.split(', ').str[1].str.split('.').str[0]
df[['Title', 'Name']].head()

In [ ]:
# survival rate and passenger count for each title
# (select the column first: pandas 3 cannot average text columns)
rate = df.groupby('Title')['Survived'].agg(['mean', 'size'])
rate.sort_values('size', ascending=False)

In [ ]:
fig = px.bar(rate.sort_values('size').reset_index(), x='mean', y='Title', orientation='h',
             hover_data=['size'], labels={'mean': 'survival rate'})
fig.show()

In [ ]:
# a new 0/1 column from the title, in one step
# (the old two-step df['Is_Married'].loc[...] = 1 changes nothing in pandas 3)
df['Is_Married'] = (df['Title'] == 'Mrs').astype(int)
df['Is_Married'].sum()